# A1 · Cleaning Real Data

*Path A: Write It Yourself, notebook 1 of 6 · about 20 to 30 minutes*

**In this notebook you will learn to:**

- Get to know three linked tables of health record data
- Tidy up messy categories with `if_else()` and `case_when()`
- Check your work after every change

**Before this notebook:** the six Foundations notebooks, ending with [F6: Combining Tables](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/F6_Combining_Tables.ipynb).

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. The project

Over the next few notebooks you will carry out a small research project from start to finish, the way you would with *All of Us* data:

> **Is taking an anticholinergic medication associated with developing dementia?**

You have three tables of **simulated** data (randomly generated, completely fake) for 100 people. The outcome is recorded as either dementia or mild cognitive impairment (MCI), an earlier stage of memory loss. In this project both count, and from here on the notebooks call them "the condition".

Before any analysis, the tables have to be cleaned and combined into one table with one row per person. This preparation is called **data wrangling**, and in real projects it takes most of the time.

Run the setup cell. The lines printed by `library(tidyverse)` and `read_csv()` are normal.

In [ ]:
library(tidyverse)

person_df    <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/demographic_data.csv")
condition_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/conditions.csv")
drug_df      <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/drug_exposures.csv")

| Table | One row is... | Columns |
|---|---|---|
| `person_df` | one person | ID, year of birth, race, sex at birth |
| `condition_df` | one diagnosis record | ID, a code for the condition, the date, the condition's name |
| `drug_df` | one prescription record | ID, a code for the drug, the date |

All three share the column `person_id`, which is how they will be linked.

## 2. Look before you touch

Start every project by looking at each table.

In [ ]:
dim(person_df)

In [ ]:
head(person_df)

**Your turn 1.** Look at `condition_df` in the same way: its size and its first rows. Then use `count()` on the column `condition_source_value` to see which conditions are recorded.

*Expected result: 96 rows and 4 columns; two conditions, Dementia (46 records) and MCI (50 records)*

In [ ]:
# Your code here

## 3. Messy categories

Look at the categories in the `race` column.

In [ ]:
person_df %>% count(race)

Seven categories, some of them very small. For analysis, you might decide to:

- shorten "Black or African American" to "Black",
- relabel "Skip" (people who skipped the question) as "Unknown",
- combine "More than one population" and "None of these" into "Other".

Changing the values in a column like this is called **recoding**.

## 4. Recoding one value: `if_else()`

You met `if_else()` in notebook F5: a test, the value to use if it is true, the value to use if it is false. To change just one category, the "false" value is simply the column itself, meaning "otherwise, leave it as it is".

In [ ]:
person_df %>%
  mutate(race = if_else(race == "Skip", "Unknown", race)) %>%
  count(race)

"Skip" has become "Unknown" and everything else is untouched. Notice that this was not stored anywhere: the result went straight into `count()` so you could look at it. `person_df` itself is unchanged. Trying a change this way before you commit to it is a good habit.

## 5. Recoding several values: `case_when()`

For several changes at once, use `case_when()`. It is a list of rules. Each rule is written `test ~ new value`: the test on the left, a squiggle `~` (called a tilde), and the value to use on the right. R goes down the list and uses the first rule whose test is true.

In [ ]:
person_df %>%
  mutate(
    race = case_when(
      race == "Black or African American" ~ "Black",
      race == "Skip" ~ "Unknown",
      race %in% c("More than one population", "None of these") ~ "Other",
      .default = race
    )
  ) %>%
  count(race)

Two new things appear here:

- **`%in%`** tests whether a value is any one of a list. `race %in% c("A", "B")` is true when race is "A" or race is "B".
- **`.default = race`** says what to do with values that match none of the rules: keep the original value.

### What if you forget `.default`?

Run this version, which leaves it out.

In [ ]:
person_df %>%
  mutate(
    race = case_when(
      race == "Black or African American" ~ "Black",
      race == "Skip" ~ "Unknown",
      race %in% c("More than one population", "None of these") ~ "Other"
    )
  ) %>%
  count(race)

62 people now have `NA` for race. Everyone who was "Asian", "White", or already "Other" matched none of the rules, and without `.default`, R filled in `NA`. **No error, no warning.** The only way to catch it is to count the categories afterwards. That is why you check.

(Nothing was stored, so `person_df` is still unchanged. If you come back and run this cell again after the next step, the count will be different, because `person_df` will already have been recoded.)

### Make the change for real

The version with `.default` gave the right result, so now store it back into `person_df`.

In [ ]:
person_df <- person_df %>%
  mutate(
    race = case_when(
      race == "Black or African American" ~ "Black",
      race == "Skip" ~ "Unknown",
      race %in% c("More than one population", "None of these") ~ "Other",
      .default = race
    )
  )

## 6. Check your work

After any recoding, check three things.

In [ ]:
person_df %>% count(race)   # 1. Are the categories what you intended?

In [ ]:
nrow(person_df)             # 2. Is the number of rows unchanged?

In [ ]:
sum(is.na(person_df$race))  # 3. Were any missing values created?

Five categories, still 100 rows, no missing values. The 15 people in "Other" are the original 4, plus 6, plus 5.

## 7. Your turn: sex at birth

In [ ]:
person_df %>% count(sex_at_birth)

**Your turn 2.** Recode `sex_at_birth` so that "Intersex", "Prefer not to answer", and "Skip" all become "Other", and "Female" and "Male" stay as they are. Follow the same steps as above:

1. Try it first without storing, ending the pipeline with `count(sex_at_birth)`.
2. When the result is right, store it back into `person_df`.

*Expected result: Female 44, Male 48, Other 8*

In [ ]:
# Your code here

**Your turn 3.** Check your recoding: confirm `person_df` still has 100 rows, and use `colSums(is.na(person_df))` to confirm that no column has missing values.

*Expected result: 100 rows; 0 missing values in every column*

In [ ]:
# Your code here

> **A note if you read other people's code:** there are several ways to recode in R. The newest versions of the tidyverse add `replace_values()`, which is used on the [workshop website](https://icohr-lab.github.io/aou-workshop/session-3.html). The result is the same, but the function may not be available in your Colab session yet.

---
## Recap

| Code | What it does |
|---|---|
| `if_else(x == "a", "b", x)` | Change "a" to "b"; leave everything else |
| `case_when(test ~ value, ...)` | A list of rules; the first true one is used |
| `.default = x` | Inside `case_when()`: what to use when no rule matches |
| `x %in% c("a", "b")` | Is x any one of these values? |
| `df %>% count(x)` | Check the categories before and after recoding |
| `colSums(is.na(df))` | Missing values in every column |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Your turn 1**

Both are treated as the outcome of interest, "the condition".

```r
dim(condition_df)
head(condition_df)
condition_df %>% count(condition_source_value)
```

**Your turn 2**

```r
person_df <- person_df %>%
  mutate(
    sex_at_birth = case_when(
      sex_at_birth %in% c("Intersex", "Prefer not to answer", "Skip") ~ "Other",
      .default = sex_at_birth
    )
  )
person_df %>% count(sex_at_birth)
```

**Your turn 3**

```r
nrow(person_df)
colSums(is.na(person_df))
```

---
## Next

Continue with [A2: One Row per Person](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/A2_One_Row_per_Person.ipynb).